# Granite 3.2 + Self-Consistency Evaluation for CLARITY

This notebook evaluates IBM Granite 3.2-2B-Instruct with self-consistency on the CLARITY dataset.

## Features:
- Uses Granite 3.2 with reasoning capabilities ()
- Self-consistency with 5 samples and majority voting
- Balanced test data loading (equal samples per label)
- Evaluation on QEvasion test set
- CLARITY submission generation

In [1]:
# Install required packages
!pip install -q transformers torch datasets pandas scikit-learn

In [2]:
# Import all necessary modules
import pickle
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


"""
Granite CLARITY Strategy

Uses IBM Granite 3.2-2B-Instruct with reasoning capabilities for CLARITY classification.
Generates JSON-structured output with reasoning and label prediction.
"""

import json
import re
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


# Global variables for Granite model (lazy loading)
_granite_model = None
_granite_tokenizer = None
_granite_device = None


def _get_device():
    """Determine the best available device."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


def _load_granite_model():
    """Lazy load Granite model and tokenizer."""
    global _granite_model, _granite_tokenizer, _granite_device
    
    if _granite_model is None:
        model_name = "ibm-granite/granite-3.2-2b-instruct"
        _granite_device = _get_device()
        
        print(f"Loading Granite model: {model_name} on {_granite_device}")
        _granite_tokenizer = AutoTokenizer.from_pretrained(model_name)
        _granite_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        _granite_model.eval()
        print("Granite model loaded successfully")
    
    return _granite_model, _granite_tokenizer, _granite_device


def call_granite_model(messages, temperature=0.7, max_new_tokens=512):
    """
    Call IBM Granite 3.2-2B-Instruct model with reasoning capabilities.
    
    Args:
        messages: List of message dicts with 'role' and 'content' keys
        temperature: Sampling temperature (default 0.7 for self-consistency)
        max_new_tokens: Maximum tokens to generate
    
    Returns:
        Generated text string (includes reasoning if thinking=True)
    """
    model, tokenizer, device = _load_granite_model()
    
    # Apply chat template with thinking=True for reasoning
    try:
        # Try to use thinking=True if supported by the tokenizer
        # Granite 3.2 supports reasoning via thinking parameter
        try:
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                thinking=True  # Enable reasoning mode
            )
        except TypeError:
            # Fallback if thinking parameter not supported
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        
        inputs = tokenizer(formatted, return_tensors="pt").to(device)
        
    except Exception as e:
        # Fallback: manual formatting
        if isinstance(messages, list) and len(messages) > 0:
            # Try to extract user message
            user_msg = None
            for msg in messages:
                if msg.get("role") == "user":
                    user_msg = msg.get("content", "")
                    break
            
            if user_msg:
                formatted = user_msg
            else:
                formatted = str(messages[-1].get("content", ""))
        else:
            formatted = str(messages)
        
        inputs = tokenizer(formatted, return_tensors="pt").to(device)
    
    # Generate with reasoning
    with torch.no_grad():
        # Granite 3.2 supports thinking via special tokens or generation config
        # We'll use the standard generation and let the model use its reasoning
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the response
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    return generated_text.strip()


def call_granite_model_batch(messages_list, temperature=0.7, max_new_tokens=512):
    """
    Batched version optimized for 15GB T4 GPU - processes multiple samples at once.
    
    Args:
        messages_list: List of message lists (one per sample)
        temperature: Sampling temperature
        max_new_tokens: Maximum tokens to generate
    
    Returns:
        List of generated text strings
    """
    model, tokenizer, device = _load_granite_model()
    
    # Format all prompts
    formatted_texts = []
    for messages in messages_list:
        try:
            try:
                formatted = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, thinking=True
                )
            except TypeError:
                formatted = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            formatted_texts.append(formatted)
        except Exception:
            # Fallback: extract user message
            user_msg = next((m.get(\"content\", \"\") for m in messages if m.get(\"role\") == \"user\"), \"\")
            formatted_texts.append(user_msg)
    
    # Tokenize batch with padding
    inputs = tokenizer(formatted_texts, return_tensors=\"pt\", padding=True, truncation=True).to(device)
    
    # Generate batch
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode each response
    results = []
    for i, output in enumerate(outputs):
        input_len = inputs.input_ids[i].shape[0]
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        results.append(generated.strip())
    
    return results


class GraniteClarityStrategy:
    """Strategy for CLARITY classification using Granite 3.2 with reasoning."""

    name = "granite-clarity"

    def build_prompt(self, question, answer):
        """
        Build prompt for CLARITY classification.
        
        Args:
            question: Interview question string
            answer: Interview answer string
        
        Returns:
            Formatted prompt string
        """
        prompt = f"""You are analyzing political interview answers for clarity classification.

Question: {question}
Answer: {answer}

Analyze the answer step-by-step:
1. Does it directly address the question?
2. Is it evasive or indirect?
3. Does it decline to answer?

Provide your reasoning and then classify as one of:
- "Direct Reply": Directly answers the question
- "Direct Non-Reply": Explicitly declines or claims inability to answer
- "Indirect": Evasive, indirect, or partially answers

Respond in JSON format:
{{
  "reasoning": "Your step-by-step analysis...",
  "label": "Direct Reply|Direct Non-Reply|Indirect"
}}"""
        return prompt

    def extract_json(self, text: str):
        """Extract JSON from Granite model response."""
        text = text.strip()

        # Remove markdown code blocks if present
        if text.startswith("```json"):
            text = text[7:]
        elif text.startswith("```"):
            text = text[3:]
        if text.endswith("```"):
            text = text[:-3]

        text = text.strip()

        # Try to find JSON object
        try:
            # First, try direct parsing
            return json.loads(text)
        except json.JSONDecodeError:
            # Try to find JSON object in the text
            json_match = re.search(r'\{[^{}]*"reasoning"[^{}]*"label"[^{}]*\}', text, re.DOTALL)
            if json_match:
                try:
                    return json.loads(json_match.group())
                except json.JSONDecodeError:
                    pass

            # Try to find any JSON object
            json_match = re.search(r'\{.*\}', text, re.DOTALL)
            if json_match:
                try:
                    return json.loads(json_match.group())
                except json.JSONDecodeError:
                    pass

            # Last resort: try to extract label and reasoning separately
            label_match = re.search(r'"label"\s*:\s*"([^"]+)"', text)
            reasoning_match = re.search(r'"reasoning"\s*:\s*"([^"]+)"', text, re.DOTALL)
            
            if label_match:
                result = {"label": label_match.group(1)}
                if reasoning_match:
                    result["reasoning"] = reasoning_match.group(1)
                else:
                    # Try to extract reasoning without quotes (might be multiline)
                    reasoning_match = re.search(r'"reasoning"\s*:\s*([^,}]+)', text, re.DOTALL)
                    if reasoning_match:
                        result["reasoning"] = reasoning_match.group(1).strip().strip('"')
                    else:
                        result["reasoning"] = "No reasoning provided"
                return result

            raise ValueError(f"Could not extract valid JSON from response: {text[:200]}...")

    def predict_single(self, question, answer, temperature=0.7):
        """
        Predict CLARITY label for a single question-answer pair.
        
        Args:
            question: Interview question
            answer: Interview answer
            temperature: Sampling temperature
        
        Returns:
            Dict with 'label' and 'reasoning' keys, or None if parsing fails
        """
        prompt = self.build_prompt(question, answer)
        
        messages = [
            {"role": "user", "content": prompt}
        ]
        
        try:
            response = call_granite_model(messages, temperature=temperature)
            parsed = self.extract_json(response)
            
            # Validate label
            valid_labels = ["Direct Reply", "Direct Non-Reply", "Indirect"]
            if parsed.get("label") not in valid_labels:
                # Try to normalize the label
                label_lower = parsed.get("label", "").lower()
                if "direct" in label_lower and "reply" in label_lower:
                    parsed["label"] = "Direct Reply"
                elif "direct" in label_lower and ("non" in label_lower or "decline" in label_lower):
                    parsed["label"] = "Direct Non-Reply"
                elif "indirect" in label_lower or "evasive" in label_lower:
                    parsed["label"] = "Indirect"
                else:
                    # Default fallback
                    parsed["label"] = "Indirect"
            
            return parsed
        except Exception as e:
            print(f"Error in predict_single: {e}")
            return None

    def predict_batch(self, examples):
        """
        Predict CLARITY labels for a batch of examples.
        
        Args:
            examples: List of dicts with 'question' and 'answer' keys
        
        Returns:
            List of dicts with 'label' and 'reasoning' keys
        """
        results = []
        for example in examples:
            question = example.get("question", "")
            answer = example.get("answer", "")
            
            if not question or not answer:
                results.append({"label": "Indirect", "reasoning": "Missing question or answer"})
                continue
            
            prediction = self.predict_single(question, answer, temperature=0.0)
            if prediction:
                results.append(prediction)
            else:
                results.append({"label": "Indirect", "reasoning": "Prediction failed"})
        
        return results

# Import our custom modules
"""
Granite Self-Consistency Strategy for CLARITY

Implements self-consistency by sampling multiple predictions and voting on labels.
Uses GraniteClarityStrategy as the base strategy.
"""

from collections import Counter


class GraniteSelfConsistencyStrategy(GraniteClarityStrategy):
    """Self-consistency wrapper for Granite CLARITY classification."""

    name = "granite-self-consistency"

    def __init__(self, samples=3, temperature=0.7):
        """
        Initialize self-consistency strategy.
        
        Args:
            samples: Number of samples to generate for voting (default: 5)
            temperature: Sampling temperature for diversity (default: 0.7)
        """
        super().__init__()
        self.samples = samples
        self.temperature = temperature

    def predict_single_with_voting(self, question, answer):
        """
        Predict label using self-consistency voting.
        
        Args:
            question: Interview question
            answer: Interview answer
        
        Returns:
            Dict with:
                - 'label': Final voted label
                - 'reasoning': All reasoning traces (list)
                - 'votes': Vote counts for each label
                - 'all_predictions': All individual predictions
        """
        predictions = []
        reasonings = []
        
        # Micro-batching: process 2 samples at a time (safe for 15GB T4 GPU)
        batch_size = 2
        prompt = self.build_prompt(question, answer)
        
        for batch_start in range(0, self.samples, batch_size):
            batch_end = min(batch_start + batch_size, self.samples)
            batch_messages = [
                [{"role": "user", "content": prompt}]
                for _ in range(batch_end - batch_start)
            ]
            
            try:
                # Generate batch
                batch_responses = call_granite_model_batch(
                    batch_messages, 
                    temperature=self.temperature
                )
                
                # Parse each response
                for response in batch_responses:
                    try:
                        parsed = self.extract_json(response)
                        # Validate and normalize label
                        valid_labels = ["Direct Reply", "Direct Non-Reply", "Indirect"]
                        if parsed.get("label") not in valid_labels:
                            label_lower = parsed.get("label", "").lower()
                            if "direct" in label_lower and "reply" in label_lower:
                                parsed["label"] = "Direct Reply"
                            elif "direct" in label_lower and ("non" in label_lower or "decline" in label_lower):
                                parsed["label"] = "Direct Non-Reply"
                            elif "indirect" in label_lower or "evasive" in label_lower:
                                parsed["label"] = "Indirect"
                            else:
                                parsed["label"] = "Indirect"
                        
                        predictions.append(parsed["label"])
                        reasonings.append(parsed.get("reasoning", "No reasoning"))
                    except Exception as e:
                        predictions.append("Indirect")
                        reasonings.append(f"Parse error: {str(e)}")
            except Exception as e:
                # Fallback to sequential if batch fails (memory issue or other error)
                print(f"Batch generation failed, falling back to sequential: {e}")
                for i in range(batch_start, batch_end):
                    try:
                        pred = self.predict_single(question, answer, temperature=self.temperature)
                        if pred and pred.get("label"):
                            predictions.append(pred["label"])
                            reasonings.append(pred.get("reasoning", "No reasoning"))
                        else:
                            predictions.append("Indirect")
                            reasonings.append("Prediction failed")
                    except Exception as e2:
                        predictions.append("Indirect")
                        reasonings.append(f"Error: {str(e2)}")
        
        # Count votes
        vote_counts = Counter(predictions)
        most_common = vote_counts.most_common(1)
        
        if most_common:
            final_label = most_common[0][0]
        else:
            final_label = "Indirect"
        
        return {
            "label": final_label,
            "reasoning": reasonings,
            "votes": dict(vote_counts),
            "all_predictions": predictions
        }

    def predict_batch(self, examples):
        """
        Predict labels for a batch using self-consistency.
        
        Args:
            examples: List of dicts with 'question' and 'answer' keys
        
        Returns:
            List of dicts with 'label', 'reasoning', 'votes', 'all_predictions'
        """
        results = []
        
        for idx, example in enumerate(examples):
            question = example.get("question", "")
            answer = example.get("answer", "")
            
            if not question or not answer:
                results.append({
                    "label": "Indirect",
                    "reasoning": ["Missing question or answer"],
                    "votes": {"Indirect": self.samples},
                    "all_predictions": ["Indirect"] * self.samples
                })
                continue
            
            print(f"Processing example {idx+1}/{len(examples)}: {question[:50]}...")
            result = self.predict_single_with_voting(question, answer)
            results.append(result)
        
        return results
"""
Balanced Dataset Loader for CLARITY

Provides utilities for loading balanced test/validation datasets
with equal representation across all three CLARITY labels.
"""

import random
from collections import Counter
from typing import List, Dict, Tuple, Optional
import pandas as pd
from datasets import load_dataset


def stratified_sample(data: List[Dict], label_key: str, samples_per_label: Optional[int] = None) -> List[Dict]:
    """
    Sample data with equal representation per label.
    
    Args:
        data: List of dicts with label_key
        label_key: Key to extract label from each dict
        samples_per_label: Number of samples per label (None = use minimum count)
    
    Returns:
        Balanced list of samples
    """
    # Group by label
    label_groups = {}
    for item in data:
        label = item.get(label_key)
        if label is None:
            continue
        if label not in label_groups:
            label_groups[label] = []
        label_groups[label].append(item)
    
    # Determine samples per label
    if samples_per_label is None:
        # Use minimum count across all labels
        counts = [len(items) for items in label_groups.values()]
        if not counts:
            return []
        samples_per_label = min(counts)
    
    # Sample equally from each label
    balanced_data = []
    for label, items in label_groups.items():
        # Sample without replacement
        sampled = random.sample(items, min(samples_per_label, len(items)))
        balanced_data.extend(sampled)
    
    return balanced_data


def load_balanced_test_data(
    split: str = "test",
    samples_per_label: Optional[int] = None,
    dataset_name: str = "ailsntua/QEvasion"
) -> Tuple[List[Dict], List[str]]:
    """
    Load balanced test data from QEvasion dataset.
    
    Args:
        split: Dataset split to use ("test" or "train")
        samples_per_label: Number of samples per label (None = use minimum)
        dataset_name: HuggingFace dataset name
    
    Returns:
        Tuple of (examples, labels) where examples are dicts with 'question' and 'answer'
    """
    print(f"Loading {split} split from {dataset_name}...")
    dataset = load_dataset(dataset_name)
    
    if split not in dataset:
        raise ValueError(f"Split '{split}' not found in dataset. Available: {list(dataset.keys())}")
    
    split_data = dataset[split]
    
    # Convert to list of dicts
    examples = []
    for item in split_data:
        # Map clarity_label to CLARITY format
        clarity_label = item.get("clarity_label", "")
        
        # Map QEvasion labels to CLARITY format
        label_mapping = {
            "Clear Reply": "Direct Reply",
            "Clear Non-Reply": "Direct Non-Reply",
            "Ambivalent Reply": "Indirect",
            "Ambivalent": "Indirect",
        }
        
        mapped_label = label_mapping.get(clarity_label, clarity_label)
        
        # Only include valid CLARITY labels
        if mapped_label not in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
            continue
        
        examples.append({
            "question": str(item.get("interview_question", item.get("question", ""))),
            "answer": str(item.get("interview_answer", "")),
            "clarity_label": mapped_label,
            "original_label": clarity_label
        })
    
    print(f"Loaded {len(examples)} examples from {split} split")
    
    # Show label distribution before balancing
    labels_before = [ex["clarity_label"] for ex in examples]
    label_counts = Counter(labels_before)
    print(f"Label distribution before balancing: {dict(label_counts)}")
    
    # Balance the dataset
    balanced_examples = stratified_sample(examples, "clarity_label", samples_per_label)
    
    # Show label distribution after balancing
    labels_after = [ex["clarity_label"] for ex in balanced_examples]
    label_counts_after = Counter(labels_after)
    print(f"Label distribution after balancing: {dict(label_counts_after)}")
    print(f"Total balanced samples: {len(balanced_examples)}")
    
    # Extract labels
    labels = [ex["clarity_label"] for ex in balanced_examples]
    
    return balanced_examples, labels


def load_clarity_eval_data(
    eval_file: str = "/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv"
) -> Tuple[List[Dict], List[int]]:
    """
    Load CLARITY evaluation dataset (no labels).
    
    Args:
        eval_file: Path to evaluation CSV file
    
    Returns:
        Tuple of (examples, indices) where examples are dicts with 'question' and 'answer'
    """
    print(f"Loading CLARITY evaluation dataset from {eval_file}...")
    
    try:
        df = pd.read_csv(eval_file)
    except FileNotFoundError:
        raise FileNotFoundError(f"Evaluation file not found: {eval_file}")
    
    examples = []
    indices = []
    
    for idx, row in df.iterrows():
        question = str(row.get("interview_question", row.get("question", ""))).strip()
        answer = str(row.get("interview_answer", "")).strip()
        
        if question and answer:
            examples.append({
                "question": question,
                "answer": answer,
                "index": idx
            })
            indices.append(idx)
    
    print(f"Loaded {len(examples)} examples from evaluation dataset")
    
    return examples, indices


def get_label_distribution(data: List[Dict], label_key: str) -> Dict[str, int]:
    """Get label distribution from data."""
    labels = [item.get(label_key) for item in data if item.get(label_key)]
    return dict(Counter(labels))

print("✅ All imports successful!")

✅ All imports successful!


## Configuration

Set parameters for evaluation:

In [9]:
# Configuration
SAMPLES = 3  # Number of self-consistency samples
TEMPERATURE = 0.7  # Sampling temperature
TEST_SAMPLES_PER_LABEL = None  # None = use minimum count across labels
MAX_TEST_SAMPLES = None  # None = evaluate all balanced samples
EVAL_FILE = "/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv"
OUTPUT_FILE = "clarity_submission_granite.pickle"

print(f"Configuration:")
print(f"  Self-consistency samples: {SAMPLES}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Test samples per label: {TEST_SAMPLES_PER_LABEL}")
print(f"  Max test samples: {MAX_TEST_SAMPLES}")

Configuration:
  Self-consistency samples: 3
  Temperature: 0.7
  Test samples per label: None
  Max test samples: None


## Initialize Strategy

In [4]:
# Initialize self-consistency strategy
strategy = GraniteSelfConsistencyStrategy(
    samples=SAMPLES,
    temperature=TEMPERATURE
)

print("✅ Strategy initialized!")

✅ Strategy initialized!


## Helper Functions

In [5]:
def map_to_clarity_format(label: str) -> str:
    """Map internal labels to CLARITY submission format."""
    mapping = {
        "Direct Reply": "Direct Reply",
        "Direct Non-Reply": "Direct Non-Reply",
        "Indirect": "Indirect",
        "Clear Reply": "Direct Reply",
        "Clear Non-Reply": "Direct Non-Reply",
        "Ambivalent Reply": "Indirect",
        "Ambivalent": "Indirect",
    }
    return mapping.get(label, "Indirect")

## Evaluate on QEvasion Test Set

In [6]:
# Load balanced test data
print("=" * 60)
print("EVALUATION ON QEVASION TEST SET")
print("=" * 60)

examples, true_labels = load_balanced_test_data(
    split="test",
    samples_per_label=TEST_SAMPLES_PER_LABEL
)

# Limit samples if requested
if MAX_TEST_SAMPLES and len(examples) > MAX_TEST_SAMPLES:
    print(f"Limiting evaluation to {MAX_TEST_SAMPLES} samples")
    examples = examples[:MAX_TEST_SAMPLES]
    true_labels = true_labels[:MAX_TEST_SAMPLES]

# Sample 2 random examples per label and print prompts BEFORE interleaving
print("\n" + "=" * 60)
print("SAMPLE EXAMPLES WITH PROMPTS (2 per label)")
print("=" * 60)

from collections import defaultdict
import random

# Group examples by label for sampling
examples_by_label = defaultdict(list)
for ex, label in zip(examples, true_labels):
    examples_by_label[label].append(ex)

# Create temporary strategy just for prompt building
strategy_temp = GraniteClarityStrategy()

# Sample 2 random examples per label and show prompts
for label in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
    if label in examples_by_label and len(examples_by_label[label]) >= 2:
        sampled = random.sample(examples_by_label[label], 2)
        print(f"\n--- {label} Examples ---")
        for idx, example in enumerate(sampled, 1):
            question = example.get("question", "")
            answer = example.get("answer", "")
            prompt = strategy_temp.build_prompt(question, answer)
            print(f"\nExample {idx}:")
            print(f"Question: {question[:150]}...")
            print(f"Answer: {answer[:150]}...")
            print(f"\nPrompt:")
            print("-" * 60)
            print(prompt)
            print("-" * 60)

# Reorganize to alternate between labels (handle imbalance)
label_groups = defaultdict(list)
for ex, label in zip(examples, true_labels):
    label_groups[label].append((ex, label))

# Interleave samples from each label group
examples_interleaved = []
true_labels_interleaved = []
max_per_label = max(len(items) for items in label_groups.values())
for i in range(max_per_label):
    for label in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
        if i < len(label_groups[label]):
            ex, lbl = label_groups[label][i]
            examples_interleaved.append(ex)
            true_labels_interleaved.append(lbl)

examples = examples_interleaved
true_labels = true_labels_interleaved

print(f"Evaluating on {len(examples)} examples (interleaved by label)...")
print(f"Label distribution: {dict(Counter(true_labels))}")

EVALUATION ON QEVASION TEST SET
Loading test split from ailsntua/QEvasion...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/308 [00:00<?, ? examples/s]

Loaded 308 examples from test split
Label distribution before balancing: {'Indirect': 206, 'Direct Reply': 79, 'Direct Non-Reply': 23}
Label distribution after balancing: {'Indirect': 23, 'Direct Reply': 23, 'Direct Non-Reply': 23}
Total balanced samples: 69
Evaluating on 69 examples (interleaved by label)...
Label distribution: {'Direct Reply': 23, 'Direct Non-Reply': 23, 'Indirect': 23}


In [10]:
# Run predictions with per-sample feedback
predictions = []
pred_labels = []
pred_labels_clarity = []
true_labels_clarity = []

print("\n" + "=" * 60)
print("RUNNING PREDICTIONS WITH PER-SAMPLE FEEDBACK")
print("=" * 60)

for idx, (example, true_label) in enumerate(zip(examples, true_labels), start=1):
    # Get prediction
    question = example.get("question", "")
    answer = example.get("answer", "")
    
    if not question or not answer:
        pred = {"label": "Indirect", "reasoning": ["Missing question or answer"], "votes": {"Indirect": SAMPLES}, "all_predictions": ["Indirect"] * SAMPLES}
    else:
        pred = strategy.predict_single_with_voting(question, answer)
    
    predictions.append(pred)
    
    # Extract and map labels
    pred_label = pred["label"]
    pred_labels.append(pred_label)
    pred_label_clarity = map_to_clarity_format(pred_label)
    pred_labels_clarity.append(pred_label_clarity)
    true_label_clarity = map_to_clarity_format(true_label)
    true_labels_clarity.append(true_label_clarity)
    
    # Check if correct
    is_correct = pred_label_clarity == true_label_clarity
    status = "✅ CORRECT" if is_correct else "❌ WRONG"
    
    # Display result
    print(f"\n[{idx+1}/{len(examples)}] {status}")
    print(f"  Question: {question[:110]}...")
    print(f"  True Label: {true_label_clarity}")
    print(f"  Predicted Label: {pred_label_clarity}")
    print(f"  Votes: {pred.get('votes', {})}")
    if not is_correct:
        print(f"  ⚠️  Expected '{true_label_clarity}' but got '{pred_label_clarity}'")

print(f"\n✅ Predictions completed for {len(predictions)} examples")


RUNNING PREDICTIONS WITH PER-SAMPLE FEEDBACK

[2/69] ✅ CORRECT
  Question: Q. Thank you, Mr. President, President Hu. President Obama, with your respect and permission, because of the t...
  True Label: Direct Reply
  Predicted Label: Direct Reply
  Votes: {'Direct Reply': 5}

[3/69] ❌ WRONG
  Question: Q. It's been three days since North Korea fired those missiles. Yesterday you said you did not know the trajec...
  True Label: Direct Non-Reply
  Predicted Label: Indirect
  Votes: {'Direct Non-Reply': 2, 'Indirect': 3}
  ⚠️  Expected 'Direct Non-Reply' but got 'Indirect'

[4/69] ❌ WRONG
  Question: Q. Mr. President, critics of your proposed bill on interrogation rules say there's another important test—thes...
  True Label: Indirect
  Predicted Label: Direct Reply
  Votes: {'Direct Reply': 5}
  ⚠️  Expected 'Indirect' but got 'Direct Reply'

[5/69] ✅ CORRECT
  Question: Q. Thank you, Mr. President. I'm curious what you would say to Americans back home who've watched their 401(k)...
 

KeyboardInterrupt: 

In [ ]:
# Compute metrics
accuracy = accuracy_score(true_labels_clarity, pred_labels_clarity)
macro_f1 = f1_score(true_labels_clarity, pred_labels_clarity, average="macro")
per_class_f1 = f1_score(true_labels_clarity, pred_labels_clarity, average=None, labels=["Direct Reply", "Direct Non-Reply", "Indirect"])

# Per-label accuracy
from collections import Counter
label_accuracies = {}
for label in ["Direct Reply", "Direct Non-Reply", "Indirect"]:
    label_mask = [tl == label for tl in true_labels_clarity]
    if any(label_mask):
        label_true = [tl for tl, m in zip(true_labels_clarity, label_mask) if m]
        label_pred = [pl for pl, m in zip(pred_labels_clarity, label_mask) if m]
        label_accuracies[label] = accuracy_score(label_true, label_pred)

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"\nPer-class F1:")
print(f"  Direct Reply: {per_class_f1[0]:.4f}")
print(f"  Direct Non-Reply: {per_class_f1[1]:.4f}")
print(f"  Indirect: {per_class_f1[2]:.4f}")
print(f"\nPer-label Accuracy:")
for label, acc in label_accuracies.items():
    print(f"  {label}: {acc:.4f}")

In [ ]:
# Confusion Matrix
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(true_labels_clarity, pred_labels_clarity, labels=["Direct Reply", "Direct Non-Reply", "Indirect"])

print("Labels: [Direct Reply, Direct Non-Reply, Indirect]")
print(cm)

In [ ]:
# Classification Report
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(true_labels_clarity, pred_labels_clarity, labels=["Direct Reply", "Direct Non-Reply", "Indirect"]))

In [ ]:
# Label Distribution
print("LABEL DISTRIBUTION")
print("=" * 60)
print("True labels:", dict(Counter(true_labels_clarity)))
print("Predicted labels:", dict(Counter(pred_labels_clarity)))

In [ ]:
# Detailed Sample Predictions with Correctness
print("\n" + "=" * 60)
print("DETAILED SAMPLE PREDICTIONS")
print("=" * 60)

# Show examples from each label category
label_indices = {"Direct Reply": [], "Direct Non-Reply": [], "Indirect": []}
for i, label in enumerate(true_labels_clarity):
    if len(label_indices[label]) < 2:  # Show 2 examples per label
        label_indices[label].append(i)

for label_type, indices in label_indices.items():
    print(f"\n--- {label_type} Examples ---")
    for i in indices:
        is_correct = pred_labels_clarity[i] == true_labels_clarity[i]
        status = "✅ CORRECT" if is_correct else "❌ WRONG"
        print(f"\nExample {i+1} [{status}]:")
        print(f"  Question: {examples[i]['question'][:120]}...")
        print(f"  Answer: {examples[i]['answer'][:120]}...")
        print(f"  True Label: {true_labels_clarity[i]}")
        print(f"  Predicted Label: {pred_labels_clarity[i]}")
        print(f"  Votes: {predictions[i].get('votes', {})}")
        print(f"  All Predictions: {predictions[i].get('all_predictions', [])}")
        if predictions[i].get('reasoning'):
            print(f"  Reasoning (first sample): {predictions[i]['reasoning'][0][:250]}...")

## Generate CLARITY Submission

In [ ]:
# Load CLARITY evaluation data
print("GENERATING CLARITY SUBMISSION")
print("=" * 60)

examples_eval, indices = load_clarity_eval_data(EVAL_FILE)

print(f"Generating predictions for {len(examples_eval)} examples...")

In [ ]:
# Run predictions on evaluation set
predictions_eval = strategy.predict_batch(examples_eval)

# Extract labels and map to CLARITY format
pred_labels_eval = [map_to_clarity_format(pred["label"]) for pred in predictions_eval]

# Validate labels
valid_labels = ["Direct Reply", "Direct Non-Reply", "Indirect"]
for i, label in enumerate(pred_labels_eval):
    if label not in valid_labels:
        print(f"Warning: Invalid label '{label}' at index {i}, defaulting to 'Indirect'")
        pred_labels_eval[i] = "Indirect"

print(f"✅ Predictions completed for evaluation set")

In [ ]:
# Create submission tuple
submission = (indices, pred_labels_eval)

# Save pickle
print(f"Saving submission to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(submission, f)

print(f"✅ Submission saved: {OUTPUT_FILE}")
print(f"   Samples: {len(indices)}")
print(f"   Predictions: {len(pred_labels_eval)}")
print(f"   Label distribution: {dict(Counter(pred_labels_eval))}")

In [ ]:
# Validate submission
print(f"Validating submission file: {OUTPUT_FILE}")

with open(OUTPUT_FILE, 'rb') as f:
    indices_check, predictions_check = pickle.load(f)

# Basic validation
assert len(indices_check) == len(predictions_check), "Indices and predictions length mismatch"
assert all(isinstance(idx, int) for idx in indices_check), "All indices must be integers"
assert all(pred in ["Direct Reply", "Direct Non-Reply", "Indirect"] for pred in predictions_check), "Invalid prediction labels"

print("✅ Submission file validation passed!")
print(f"   - {len(indices_check)} predictions")
print(f"   - Indices range: {min(indices_check)} to {max(indices_check)}")
print(f"   - Unique predictions: {set(predictions_check)}")

## Summary

Evaluation complete! The submission file is ready for CLARITY-SemEval-2026.